In [2]:
!pip install sklearn

^C
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.0 MB 12.4 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.0 MB 12.4 MB/s eta 0:00:01
   -------------------------------------- - 7.6/8.0 MB 12.4 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 12.1 MB/s eta 0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   -- ------------------------------------- 2.4/36.5 MB 12.4 MB/s eta 0:00:03
   ----- ---------------------------------- 5.0/36.5 MB 12.4 MB/s eta 0:00:03
   -------- ------------------------------- 7.3/36.5 MB 12.4 MB/s eta 0:00:03
   ----------- ---------------------------- 10.2/36.5 MB 12.4 MB/s eta 0:00:03
   ------------- -------------------------- 12.6/36.5


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
# 1. 기본 시스템 및 데이터 처리
import os
import sys
import warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from getpass import getpass
import json
import duckdb
import time
import shapely  # 추가 필요

# 2. 지리 정보 및 기하학적 처리 (GIS)
import shapely as shp
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon, MultiPoint, box
from geopandas import GeoDataFrame, read_file
from geopy.distance import great_circle

# 3. 선박 궤적 분석 (MovingPandas)
import movingpandas as mpd

# 4. 머신러닝 및 데이터 분석
from sklearn.cluster import DBSCAN

# 5. 시각화 (Matplotlib & HoloViews/hvPlot)
import matplotlib.pyplot as plt
import hvplot.pandas
from holoviews import opts, dim, Layout 

# 6. 경고 메시지 무시 설정
warnings.simplefilter("ignore")

# (선택 사항) 커스텀 모듈 경로 추가
sys.path.append("..")


In [45]:
start = time.time()
ais_df = pd.read_csv("./기상데이터/Sample01_AIS_new_category_final.csv") 

min_lon, max_lon = 125.678, 131.229
max_lat = 36.001

ais_df_filtered = ais_df[
    (ais_df['longitude'] >= min_lon) & (ais_df['longitude'] <= max_lon) & (ais_df['latitude'] <= max_lat)
]

ais = ais_df_filtered

# if filtering by "timestamp" 
start_time = "2022-12-01 00:00:00"
end_time = "2022-12-02 23:50:00"
time_diff = pd.to_datetime(end_time) - pd.to_datetime(start_time)

# Dataframe을 GeoDataFrame으로 전환
ais = gpd.GeoDataFrame(
    # compressed_once_df,
    ais,
    geometry = gpd.points_from_xy(ais.longitude, ais.latitude),
    crs="EPSG:4326"
)

#시간 인덱스 설정 및 속도 0이상인 항적만 추출
ais['t'] = pd.to_datetime(ais['timestamp'], format='mixed', utc=True)
ais = ais.set_index('t')
ais = ais[(ais['timestamp']>= start_time)&(ais['timestamp'] <= end_time)]
ais = ais[ais.speed>0]

ais = ais.sort_values(by=['mmsi', 't']) # MMSI별 시간순 정렬 강제

# cluster 분석을 위한 전처리
MIN_LENGTH = 1000 # in meters; 1000m 이상인 항적들만 추출
TRIP_ID = 'mmsi'
traj_collection = mpd.TrajectoryCollection(ais, TRIP_ID, min_length=MIN_LENGTH)

trips = mpd.ObservationGapSplitter(traj_collection).split(gap=timedelta(minutes=120)) #60분 이산 정박한 경우 trajectory 분할

aggregator = mpd.TrajectoryCollectionAggregator(trips, max_distance=100000, min_distance=2000, min_stop_duration=timedelta(minutes=120))

flows = aggregator.get_flows_gdf()
clusters = aggregator.get_clusters_gdf()

################################################
### LLM 활용을 통한 항적 패턴 요약을 위한 전처리 ###
################################################

### 전체 항적 패턴 요약에 사용할 해상 구역 정의
zone_configs = {
    # --- [Layer 1: 북부 연안 및 울산] Lat: 34.70 ~ 36.00 (일부 34.40 시작) ---
    'Zone_16':   [125.00, 34.40, 125.50, 36.00, '서해 남부 외해'],
    'Zone_5':    [125.50, 34.40, 126.60, 36.00, '목포-신안 해역'],
    'Zone_12':   [126.60, 34.40, 127.60, 36.00, '고흥-완도 연안'],
    'Zone_3':    [127.60, 34.70, 128.00, 36.00, '광양-여수 해역'],
    'Zone_4':    [128.00, 34.70, 128.30, 36.00, '남해 중앙 연안'],
    'Zone_2':    [128.30, 34.70, 128.70, 36.00, '거제-통영 해역'],
    'Zone_1':    [128.70, 34.70, 129.30, 36.00, '부산-가덕 해역'],
    'Zone_18':   [129.30, 34.70, 130.50, 36.00, '부산외항-울산남부'], # 부산 동쪽 빈틈 메움
    'Zone_14':   [130.50, 35.15, 132.50, 36.00, '울산-포항 해역'],
    
    # --- [Layer 2: 중단 외해 및 대한해협] Lat: 33.80 ~ 34.70 ---
    'Zone_7':    [125.00, 33.00, 126.20, 34.40, '서남해 외해'],
    'Zone_8':    [126.20, 33.80, 127.60, 34.40, '제주-육지 교차로 해역'],
    'Zone_9_1':  [127.60, 33.80, 128.30, 34.70, '여수-통영 외해'],
    'Zone_9_2':  [128.30, 33.80, 128.70, 34.70, '거제-가덕 외해'],
    'Zone_9_3':  [128.70, 33.80, 129.30, 34.70, '부산-대마도 입구 해역'], # 부산 하단 빈틈 메움
    'Zone_15_1': [129.30, 33.80, 130.50, 34.70, '대마도 남서 해역'],
    'Zone_17':   [130.50, 33.80, 131.50, 34.25, '시모노세키-기타큐슈 해역'],
    'Zone_15_2': [130.50, 34.25, 131.50, 35.15, '대마도 북동 해역'],
    
    # --- [Layer 3: 제주 및 일본 연안] Lat: 33.00 ~ 33.80 ---
    'Zone_6':    [126.20, 33.40, 126.90, 33.80, '제주 북부 연안'],
    'Zone_13':   [126.20, 33.00, 126.90, 33.40, '제주 남부 연안'],
    'Zone_11':   [126.90, 33.00, 129.50, 33.80, '제주 남동부 해역'],
    'Zone_15_3': [129.50, 33.00, 131.00, 33.80, '이키-후쿠오카 연안'],
    'Zone_15_4': [131.00, 33.00, 132.50, 35.15, '일본 가이요 외해'],
    
    # --- [Layer 4: 최남단 원거리] Lat: 31.00 ~ 33.00 ---
    'Zone_10':   [125.00, 31.00, 132.50, 33.00, '원거리 국제공해']
}

polygons = []
ids = []
names = []

for zid, val in zone_configs.items():
    polygons.append(box(val[0], val[1], val[2], val[3]))
    ids.append(zid)
    names.append(val[4])

zones_gdf = gpd.GeoDataFrame({'zone_id': ids, 'name': names}, 
                             geometry=polygons, crs="EPSG:4326")

named_clusters = gpd.sjoin(clusters, zones_gdf, how="inner", predicate="within")

cluster_summary = named_clusters.groupby('name')['n'].agg(['count', 'sum']).rename(columns={'count':'stop_count', 'sum':'total_stay_index'}).reset_index()
cluster_summary = cluster_summary.sort_values(by='total_stay_index', ascending=False)

data_dict_cluster = cluster_summary.to_dict(orient="records")

clusters_data = json.dumps(data_dict_cluster, ensure_ascii=False, indent=4)
print(f"소요시간: {time.time()-start}초")

소요시간: 7.941018342971802초


In [15]:
type(flows)

geopandas.geodataframe.GeoDataFrame

In [5]:
print(clusters_data)

[
    {
        "name": "부산-가덕 해역",
        "stop_count": 1,
        "total_stay_index": 113
    },
    {
        "name": "시모노세키-기타큐슈 해역",
        "stop_count": 2,
        "total_stay_index": 90
    },
    {
        "name": "여수-통영 외해",
        "stop_count": 1,
        "total_stay_index": 89
    },
    {
        "name": "거제-가덕 외해",
        "stop_count": 1,
        "total_stay_index": 79
    },
    {
        "name": "일본 가이요 외해",
        "stop_count": 1,
        "total_stay_index": 66
    },
    {
        "name": "원거리 국제공해",
        "stop_count": 5,
        "total_stay_index": 58
    },
    {
        "name": "제주 남동부 해역",
        "stop_count": 2,
        "total_stay_index": 49
    },
    {
        "name": "부산외항-울산남부",
        "stop_count": 1,
        "total_stay_index": 49
    },
    {
        "name": "목포-신안 해역",
        "stop_count": 2,
        "total_stay_index": 36
    },
    {
        "name": "제주 북부 연안",
        "stop_count": 1,
        "total_stay_index": 23
    },
    {
        "name

In [19]:
start = time.time()

# 1. DuckDB 파일 연결 및 공간 확장 로드
db_path = r"D:\LEE\AI_team\github\Vision_AI_RnD_team\projects\test_project\ais_weather\test.db"
con = duckdb.connect(database=db_path)
con.execute("INSTALL spatial; LOAD spatial;")

table_name = "AIS_category"

# 2. 초기 데이터 필터링 (DuckDB)
# CSV를 읽으면서 위경도/시간/속도를 즉시 필터링
min_lon, max_lon, max_lat = 125.678, 131.229, 36.001
start_time, end_time = "2022-12-01 00:00:00", "2022-12-02 23:50:00"

# 쿼리 실행 (DB 내부에서 직접 필터링)
query = f"""
    SELECT *, ST_AsWKB(ST_Point(longitude, latitude)) as geom_wkb, CAST(timestamp AS TIMESTAMP) as t
    FROM {table_name}
    WHERE longitude >= {min_lon} AND longitude <= {max_lon}
      AND latitude <= {max_lat}
      AND timestamp >= '{start_time}' 
      AND timestamp <= '{end_time}'
      AND speed > 0
    ORDER BY MMSI, t ASC  -- 이 줄이 필수입니다!
"""

# 3. DuckDB 결과를 Pandas로 가져오기
df = con.execute(query).df()

# 4. WKB 컬럼을 이용해 즉시 GeoDataFrame 생성
# 이 방식은 points_from_xy보다 대용량 처리 시 훨씬 빠릅니다.
ais_gdf = gpd.GeoDataFrame(
    df, 
    geometry=shapely.from_wkb(df['geom_wkb'].apply(bytes)),
    crs="EPSG:4326"
).set_index('t')

# 3. MovingPandas 집계 (클러스터 추출)
traj_collection = mpd.TrajectoryCollection(ais_gdf, 'mmsi', min_length=1000)
trips = mpd.ObservationGapSplitter(traj_collection).split(gap=timedelta(minutes=120))
aggregator = mpd.TrajectoryCollectionAggregator(
    trips, max_distance=100000, min_distance=2000, min_stop_duration=timedelta(minutes=120)
)
clusters_gdf = aggregator.get_clusters_gdf()

### 전체 항적 패턴 요약에 사용할 해상 구역 정의
zone_configs = {
    # --- [Layer 1: 북부 연안 및 울산] Lat: 34.70 ~ 36.00 (일부 34.40 시작) ---
    'Zone_16':   [125.00, 34.40, 125.50, 36.00, '서해 남부 외해'],
    'Zone_5':    [125.50, 34.40, 126.60, 36.00, '목포-신안 해역'],
    'Zone_12':   [126.60, 34.40, 127.60, 36.00, '고흥-완도 연안'],
    'Zone_3':    [127.60, 34.70, 128.00, 36.00, '광양-여수 해역'],
    'Zone_4':    [128.00, 34.70, 128.30, 36.00, '남해 중앙 연안'],
    'Zone_2':    [128.30, 34.70, 128.70, 36.00, '거제-통영 해역'],
    'Zone_1':    [128.70, 34.70, 129.30, 36.00, '부산-가덕 해역'],
    'Zone_18':   [129.30, 34.70, 130.50, 36.00, '부산외항-울산남부'], # 부산 동쪽 빈틈 메움
    'Zone_14':   [130.50, 35.15, 132.50, 36.00, '울산-포항 해역'],
    
    # --- [Layer 2: 중단 외해 및 대한해협] Lat: 33.80 ~ 34.70 ---
    'Zone_7':    [125.00, 33.00, 126.20, 34.40, '서남해 외해'],
    'Zone_8':    [126.20, 33.80, 127.60, 34.40, '제주-육지 교차로 해역'],
    'Zone_9_1':  [127.60, 33.80, 128.30, 34.70, '여수-통영 외해'],
    'Zone_9_2':  [128.30, 33.80, 128.70, 34.70, '거제-가덕 외해'],
    'Zone_9_3':  [128.70, 33.80, 129.30, 34.70, '부산-대마도 입구 해역'], # 부산 하단 빈틈 메움
    'Zone_15_1': [129.30, 33.80, 130.50, 34.70, '대마도 남서 해역'],
    'Zone_17':   [130.50, 33.80, 131.50, 34.25, '시모노세키-기타큐슈 해역'],
    'Zone_15_2': [130.50, 34.25, 131.50, 35.15, '대마도 북동 해역'],
    
    # --- [Layer 3: 제주 및 일본 연안] Lat: 33.00 ~ 33.80 ---
    'Zone_6':    [126.20, 33.40, 126.90, 33.80, '제주 북부 연안'],
    'Zone_13':   [126.20, 33.00, 126.90, 33.40, '제주 남부 연안'],
    'Zone_11':   [126.90, 33.00, 129.50, 33.80, '제주 남동부 해역'],
    'Zone_15_3': [129.50, 33.00, 131.00, 33.80, '이키-후쿠오카 연안'],
    'Zone_15_4': [131.00, 33.00, 132.50, 35.15, '일본 가이요 외해'],
    
    # --- [Layer 4: 최남단 원거리] Lat: 31.00 ~ 33.00 ---
    'Zone_10':   [125.00, 31.00, 132.50, 33.00, '원거리 국제공해']
}

polygons = []
ids = []
names = []

for zid, val in zone_configs.items():
    polygons.append(box(val[0], val[1], val[2], val[3]))
    ids.append(zid)
    names.append(val[4])

zones_gdf = gpd.GeoDataFrame({'zone_id': ids, 'name': names}, 
                             geometry=polygons, crs="EPSG:4326")

# 4. DuckDB 공간 조인 및 최종 통계 처리
# Geometry를 DuckDB가 이해할 수 있는 WKB로 변환하여 등록
clusters_gdf['geom_wkb'] = clusters_gdf['geometry'].to_wkb()
zones_gdf['geom_wkb'] = zones_gdf['geometry'].to_wkb()

con.register('v_clusters', clusters_gdf[['n', 'geom_wkb']])
con.register('v_zones', zones_gdf[['name', 'geom_wkb']])


# [공간 조인 + 그룹화 + 정렬]을 한 번의 SQL로 처리
# ST_GeomFromWKB를 사용하여 바이너리를 공간 객체로 복원 후 연산
final_query = """
    SELECT z.name, COUNT(*) as stop_count, SUM(CAST(c.n AS DOUBLE)) as total_stay_index
    FROM v_clusters c, v_zones z
    WHERE ST_Intersects(ST_GeomFromWKB(c.geom_wkb), ST_GeomFromWKB(z.geom_wkb))
    GROUP BY z.name
    ORDER BY total_stay_index DESC
"""

# 5. 결과 추출 및 JSON 변환
final_summary_df = con.execute(final_query).df()

clusters_data_json = json.dumps(
    final_summary_df.to_dict(orient="records"), 
    ensure_ascii=False, 
    indent=4
)

print(f"소요시간: {time.time()-start}초")

소요시간: 5.838400363922119초


In [42]:
type(aggregator)

movingpandas.trajectory_aggregator.TrajectoryCollectionAggregator

In [93]:
print(clusters_data_json)

[
    {
        "name": "부산-가덕 해역",
        "stop_count": 1,
        "total_stay_index": 113.0
    },
    {
        "name": "시모노세키-기타큐슈 해역",
        "stop_count": 2,
        "total_stay_index": 90.0
    },
    {
        "name": "여수-통영 외해",
        "stop_count": 1,
        "total_stay_index": 89.0
    },
    {
        "name": "거제-가덕 외해",
        "stop_count": 1,
        "total_stay_index": 79.0
    },
    {
        "name": "일본 가이요 외해",
        "stop_count": 1,
        "total_stay_index": 66.0
    },
    {
        "name": "원거리 국제공해",
        "stop_count": 5,
        "total_stay_index": 58.0
    },
    {
        "name": "제주 남동부 해역",
        "stop_count": 2,
        "total_stay_index": 49.0
    },
    {
        "name": "부산외항-울산남부",
        "stop_count": 1,
        "total_stay_index": 49.0
    },
    {
        "name": "목포-신안 해역",
        "stop_count": 2,
        "total_stay_index": 36.0
    },
    {
        "name": "제주 북부 연안",
        "stop_count": 1,
        "total_stay_index": 23.0
    },

In [46]:
start = time.time()
ais_df = pd.read_csv("./기상데이터/Sample01_AIS_new_category_final.csv") 

min_lon, max_lon = 125.678, 131.229
max_lat = 36.001

ais_df_filtered = ais_df[
    (ais_df['longitude'] >= min_lon) & (ais_df['longitude'] <= max_lon) & (ais_df['latitude'] <= max_lat)
]


ais = ais_df_filtered

# if filtering by "timestamp" 
start_time = "2022-12-01 00:00:00"
end_time = "2022-12-02 23:50:00"
time_diff = pd.to_datetime(end_time) - pd.to_datetime(start_time)

# Dataframe을 GeoDataFrame으로 전환
ais = gpd.GeoDataFrame(
    # compressed_once_df,
    ais,
    geometry = gpd.points_from_xy(ais.longitude, ais.latitude),
    crs="EPSG:4326"
)

#시간 인덱스 설정 및 속도 0이상인 항적만 추출
ais['t'] = pd.to_datetime(ais['timestamp'], format='mixed', utc=True)
ais = ais.set_index('t')
ais = ais[(ais['timestamp']>= start_time)&(ais['timestamp'] <= end_time)]
ais = ais[ais.speed>0]

# sjoin을 통해 포인트가 속한 구역정보를 결합
joined_gdf = gpd.sjoin(ais, zones_gdf, how='left', predicate='within')
# 결합 후 구역에 속하지 않은 데이터 NaN 됨
joined_gdf = joined_gdf.dropna(subset=['name'])

# 구역별 속도 통계 산출
zone_speed_stats = joined_gdf.groupby('name')['speed'].agg(['mean', 'max', 'count']).reset_index()

# LLM에 전달하기 위해 json으로 변환
data_dict_speed = zone_speed_stats.to_dict(orient="records")
speed_data = json.dumps(data_dict_speed, ensure_ascii=False, indent=4)

print(f"소요시간: {time.time()-start}초")

소요시간: 2.4259722232818604초


In [99]:
print(speed_data)

[
    {
        "name": "거제-가덕 외해",
        "mean": 8.497797356828194,
        "max": 19.4,
        "count": 1135
    },
    {
        "name": "거제-통영 해역",
        "mean": 10.497297297297298,
        "max": 12.8,
        "count": 37
    },
    {
        "name": "고흥-완도 연안",
        "mean": 4.14,
        "max": 10.0,
        "count": 5
    },
    {
        "name": "광양-여수 해역",
        "mean": 6.16703056768559,
        "max": 15.7,
        "count": 458
    },
    {
        "name": "대마도 남서 해역",
        "mean": 4.36021897810219,
        "max": 17.2,
        "count": 1370
    },
    {
        "name": "대마도 북동 해역",
        "mean": 12.657608695652174,
        "max": 16.3,
        "count": 92
    },
    {
        "name": "목포-신안 해역",
        "mean": 8.364968152866242,
        "max": 22.7,
        "count": 157
    },
    {
        "name": "부산-가덕 해역",
        "mean": 2.370750053728777,
        "max": 102.3,
        "count": 4653
    },
    {
        "name": "부산-대마도 입구 해역",
        "mean": 11.92513287

In [102]:
start = time.time()
# 1. DuckDB 연결 및 공간 확장 로드
db_path = r"D:\AI_team\github\Vision_AI_RnD_team\projects\test_project\ais_weather\test.db"
con = duckdb.connect(database=db_path)
con.execute("INSTALL spatial; LOAD spatial;")

# 2. 기준 구역(zones_gdf)을 DuckDB에 등록
# 공간 조인을 위해 zones_gdf를 WKB로 변환하여 임시 테이블로 등록합니다.
zones_gdf['geom_wkb'] = zones_gdf['geometry'].to_wkb()
con.register('v_zones', zones_gdf[['name', 'geom_wkb']])

# 3. DuckDB 통합 쿼리 (필터링 + 공간 조인 + 통계 요약)
# 포인트 생성부터 구역 매칭, 평균/최대 속도 계산까지 SQL 한 번에 처리합니다.
min_lon, max_lon, max_lat = 125.678, 131.229, 36.001
start_time, end_time = "2022-12-01 00:00:00", "2022-12-02 23:50:00"

integrated_query = f"""
    WITH filtered_ais AS (
        -- [단계 1] 기본 필터링 및 포인트 생성
        SELECT 
            speed,
            ST_Point(longitude, latitude) as point_geom
        FROM AIS_category
        WHERE longitude BETWEEN {min_lon} AND {max_lon}
          AND latitude <= {max_lat}
          AND timestamp >= '{start_time}' 
          AND timestamp <= '{end_time}'
          AND speed > 0
    ),
    joined_data AS (
        -- [단계 2] 구역 테이블(v_zones)과 공간 조인 (ST_Within)
        -- INNER JOIN을 통해 구역에 속하지 않은(NaN이 될) 데이터는 자동 필터링됩니다.
        SELECT 
            z.name,
            a.speed
        FROM filtered_ais a
        JOIN v_zones z ON ST_Within(a.point_geom, ST_GeomFromWKB(z.geom_wkb))
    )
    -- [단계 3] 구역별 속도 통계 산출
    SELECT 
        name,
        AVG(speed) as mean,
        MAX(speed) as max,
        COUNT(*) as count
    FROM joined_data
    GROUP BY name
    ORDER BY name ASC
"""

# 4. 결과 실행 및 JSON 변환
# 데이터 요약본만 파이썬으로 넘어오므로 매우 가볍습니다.
speed_stats_df = con.execute(integrated_query).df()
speed_data = json.dumps(
    speed_stats_df.to_dict(orient="records"), 
    ensure_ascii=False, 
    indent=4
)

print(f"소요시간: {time.time()-start}초")

소요시간: 0.15985345840454102초


In [103]:
print(speed_data)

[
    {
        "name": "거제-가덕 외해",
        "mean": 8.497797356828205,
        "max": 19.4,
        "count": 1135
    },
    {
        "name": "거제-통영 해역",
        "mean": 10.497297297297298,
        "max": 12.8,
        "count": 37
    },
    {
        "name": "고흥-완도 연안",
        "mean": 4.14,
        "max": 10.0,
        "count": 5
    },
    {
        "name": "광양-여수 해역",
        "mean": 6.167030567685563,
        "max": 15.7,
        "count": 458
    },
    {
        "name": "대마도 남서 해역",
        "mean": 4.360218978102182,
        "max": 17.2,
        "count": 1370
    },
    {
        "name": "대마도 북동 해역",
        "mean": 12.657608695652167,
        "max": 16.3,
        "count": 92
    },
    {
        "name": "목포-신안 해역",
        "mean": 8.364968152866242,
        "max": 22.7,
        "count": 157
    },
    {
        "name": "부산-가덕 해역",
        "mean": 2.3707500537289263,
        "max": 102.3,
        "count": 4653
    },
    {
        "name": "부산-대마도 입구 해역",
        "mean": 11.92513

In [2]:
def compress_ship_data_duckdb_further(db_path, table_name, min_time, max_time, min_lon=125.678, max_lon=131.229, max_lat=36.001):
    db_path = db_path.replace("\\", "/") 
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL sqlite; LOAD sqlite;")
    con.execute(f"ATTACH '{db_path}' AS sqlite_db (TYPE SQLITE);")

    query = f"""
    -- [STEP 1] Raw 데이터 로드 및 1차 트리거 (정수 변환 비교로 정밀도 확보)
    WITH raw_data AS (
        SELECT *,
            CAST(timestamp AS TIMESTAMP) as ts,
            CAST(longitude AS DOUBLE) as lon_val,
            CAST(latitude AS DOUBLE) as lat_val,
            CAST(course AS DOUBLE) as c_course,
            CAST(speed AS DOUBLE) as s_speed,
            row_number() OVER () as temp_row_idx
        FROM sqlite_db.{table_name}
        WHERE longitude >= {min_lon} AND longitude <= {max_lon} AND latitude <= {max_lat}
            AND timestamp BETWEEN '{min_time}' AND '{max_time}'
    ),
    ordered_data AS (
        SELECT *,
            LAG(lon_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lon,
            LAG(lat_val) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_lat,
            LAG(c_course) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_course,
            LAG(s_speed) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx) as prev_speed
        FROM raw_data
    ),
    diff_calc AS (
        SELECT *,
            CASE WHEN CAST(lon_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lon, lon_val) * 1000 AS BIGINT) 
                   OR CAST(lat_val * 1000 AS BIGINT) != CAST(COALESCE(prev_lat, lat_val) * 1000 AS BIGINT) 
                 THEN 1 ELSE 0 END as pos_change,
            CASE 
                WHEN (c_course - COALESCE(prev_course, c_course)) > 180 THEN (c_course - COALESCE(prev_course, c_course)) - 360
                WHEN (c_course - COALESCE(prev_course, c_course)) < -180 THEN (c_course - COALESCE(prev_course, c_course)) + 360
                ELSE (c_course - COALESCE(prev_course, c_course))
            END as course_diff,
            (s_speed - COALESCE(prev_speed, s_speed)) as speed_diff
        FROM ordered_data
    ),
    event_logic AS (
        SELECT *,
            -- 경계값 오차 방지를 위해 0.000001 보정 (Pandas와의 일치성 향상)
            CASE WHEN pos_change = 1 OR (ABS(course_diff) >= 19.999999 AND s_speed >= 0.999999) OR ABS(speed_diff) >= 1.999999 THEN 1 ELSE 0 END as event_trigger
        FROM diff_calc
    ),
    grouping_v1 AS (
        SELECT *,
            SUM(event_trigger) OVER (PARTITION BY ShipName ORDER BY ts, temp_row_idx ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id_v1
        FROM event_logic
    ),
    summarized_v1 AS (
        -- [1차 압축] ANY_VALUE 대신 FIRST를 사용하여 Pandas .first()와 100% 일치화
        SELECT 
            ShipName, group_id_v1,
            FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
            MIN(ts) as start_time, MAX(ts) as end_time,
            FIRST(lon_val) as lon, FIRST(lat_val) as lat,
            FIRST(c_course) as first_course, AVG(s_speed) as avg_speed,
            FIRST(course_diff) as turn_val, FIRST(speed_diff) as accel_val
        FROM grouping_v1 
        GROUP BY ShipName, group_id_v1
    ),
    -- [STEP 2] 상태 판별 (부동소수점 오차 차단)
    status_calc AS (
        SELECT *,
            avg_speed - COALESCE(LAG(avg_speed) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), avg_speed) as group_speed_diff
        FROM summarized_v1
    ),
    status_final AS (
        SELECT *,
            CASE 
                -- 1.0, 20.0 등의 경계값을 소수점 8자리에서 반올림 후 비교하여 Pandas와 일치시킴
                WHEN ROUND(avg_speed, 8) < 1.0 THEN '정박/대기'
                ELSE TRIM(CONCAT_WS(' ',
                    CASE WHEN ROUND(ABS(turn_val), 8) >= 20.0 AND ROUND(avg_speed, 8) >= 1.0 
                         THEN (CASE WHEN turn_val > 0 THEN '우선회' ELSE '좌선회' END) || '(' || ROUND(ABS(turn_val), 1) || '°)' ELSE '' END,
                    CASE WHEN ROUND(group_speed_diff, 8) >= 2.0 THEN '가속' 
                         WHEN ROUND(group_speed_diff, 8) < -2.0 THEN '감속' ELSE '' END,
                    CASE WHEN ROUND(avg_speed, 8) >= 5.0 THEN '이동/통과' ELSE '저속 운항' END
                ))
            END as status
        FROM status_calc
    ),
    -- [STEP 3] 2차 압축
    v2_trigger AS (
        SELECT *,
            CASE WHEN status != COALESCE(LAG(status) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1), status) 
                 THEN 1 ELSE 0 END as status_change
        FROM status_final
    ),
    v2_grouping AS (
        SELECT *,
            SUM(status_change) OVER (PARTITION BY ShipName ORDER BY start_time, group_id_v1 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as group_id
        FROM v2_trigger
    )
    -- [STEP 4] 최종 요약 (집계 방식 일치)
    SELECT 
        ShipName, group_id,
        FIRST(mmsi) as mmsi, FIRST(higher_types) as higher_types, FIRST(radius) as radius,
        MIN(start_time) as start_time, MAX(end_time) as end_time,
        FIRST(lon) as lon, FIRST(lat) as lat,
        FIRST(first_course) as first_course,
        AVG(avg_speed) as avg_speed,
        FIRST(turn_val) as turn_val,
        FIRST(accel_val) as accel_val,
        FIRST(status) as status
    FROM v2_grouping
    GROUP BY ShipName, group_id
    ORDER BY ShipName, start_time
    """

    df_result = con.execute(query).df()
    con.execute("DETACH sqlite_db;")
    return df_result

In [9]:
def get_direction_analysis_duckdb(df):
    # 1. DuckDB 연결 (메모리 모드) 및 공간 확장 로드
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL spatial; LOAD spatial;")

    # 2. Pandas DataFrame을 DuckDB 테이블로 등록
    # 이렇게 하면 SQL 쿼리에서 'v_ais_data'라는 이름으로 df를 참조할 수 있습니다.
    con.register('v_ais_data', df)

    # 3. DuckDB SQL 연산 통합 (Binning + Vector Mean)
    query = """
    WITH filtered_moving AS (
        -- [단계 1] '정박/대기'를 제외하고 MMSI별 마지막 경로와 평균 속도 추출
        SELECT 
            mmsi,
            -- 가장 최근의 경로를 가져오기 위해 start_time(또는 end_time) 기준 arg_max 사용
            arg_max(first_course, start_time) as last_course,
            AVG(avg_speed) as mean_speed
        FROM v_ais_data
        WHERE status != '정박/대기'
        GROUP BY mmsi
    ),
    binned_directions AS (
        -- [단계 2] 8방위 분류 (Binning)
        SELECT 
            *,
            CASE 
                WHEN last_course >= 337.5 OR last_course < 22.5 THEN '북'
                WHEN last_course >= 22.5 AND last_course < 67.5 THEN '북동'
                WHEN last_course >= 67.5 AND last_course < 112.5 THEN '동'
                WHEN last_course >= 112.5 AND last_course < 157.5 THEN '남동'
                WHEN last_course >= 157.5 AND last_course < 202.5 THEN '남'
                WHEN last_course >= 202.5 AND last_course < 247.5 THEN '남서'
                WHEN last_course >= 247.5 AND last_course < 292.5 THEN '서'
                WHEN last_course >= 292.5 AND last_course < 337.5 THEN '북서'
            END AS direction_group
        FROM filtered_moving
    ),
    vector_agg AS (
        -- [단계 3] 방향 그룹별 통계 및 벡터 평균 연산
        SELECT 
            direction_group,
            COUNT(*) as ship_count,
            AVG(mean_speed) as average_speed_knot,
            AVG(sin(radians(last_course))) as mean_sin,
            AVG(cos(radians(last_course))) as mean_cos
        FROM binned_directions
        GROUP BY direction_group
    )
    -- [단계 4] 최종 결과 포맷팅
    SELECT 
        direction_group || '진' as direction,
        ship_count,
        ROUND(average_speed_knot, 1) as average_speed_knot,
        ROUND((degrees(atan2(mean_sin, mean_cos)) + 360) % 360, 1) as vector_course_deg
    FROM vector_agg
    ORDER BY 
        CASE direction_group 
            WHEN '북' THEN 1 WHEN '북동' THEN 2 WHEN '동' THEN 3 WHEN '남동' THEN 4 
            WHEN '남' THEN 5 WHEN '남서' THEN 6 WHEN '서' THEN 7 WHEN '북서' THEN 8 
        END
    """
    
    # 4. 결과 실행 및 JSON 변환
    result_df = con.execute(query).df()
    
    if result_df.empty:
        return json.dumps([{"message": "현재 이동 중인 주요 선박 흐름 없음"}], ensure_ascii=False)
        
    return json.dumps(result_df.to_dict(orient="records"), ensure_ascii=False, indent=4)

In [10]:
start = time.time()
testdb = "D:/LEE/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/test.db"
comp_ship_data = compress_ship_data_duckdb_further(testdb, "AIS_category", "2022-12-01 00:00:00", "2022-12-02 23:50:00")
direction_flow_data = get_direction_analysis_duckdb(comp_ship_data)
print(f"소요시간: {time.time()-start}초")

소요시간: 0.22424674034118652초


In [12]:
print(direction_flow_data)

[
    {
        "direction": "북진",
        "ship_count": 4,
        "average_speed_knot": 8.8,
        "vector_course_deg": 5.4
    },
    {
        "direction": "북동진",
        "ship_count": 5,
        "average_speed_knot": 8.6,
        "vector_course_deg": 37.1
    },
    {
        "direction": "동진",
        "ship_count": 6,
        "average_speed_knot": 8.7,
        "vector_course_deg": 91.0
    },
    {
        "direction": "남동진",
        "ship_count": 6,
        "average_speed_knot": 10.9,
        "vector_course_deg": 145.8
    },
    {
        "direction": "남진",
        "ship_count": 6,
        "average_speed_knot": 8.4,
        "vector_course_deg": 179.1
    },
    {
        "direction": "남서진",
        "ship_count": 14,
        "average_speed_knot": 9.9,
        "vector_course_deg": 227.3
    },
    {
        "direction": "서진",
        "ship_count": 9,
        "average_speed_knot": 8.2,
        "vector_course_deg": 266.7
    },
    {
        "direction": "북서진",
        "ship_coun

In [37]:
flows = aggregator.get_flows_gdf()

# 2. Flow(Line)의 시작점과 끝점에 Zone 이름 부여
# Line의 첫 번째 점과 마지막 점을  추출하여 각각 Join
flows['start_point'] = flows.geometry.apply(lambda x: x.coords[0])
flows['end_point'] = flows.geometry.apply(lambda x: x.coords[-1])

# 시작점/끝점용 임시 gdf 생성 후 Spatial Join
start_gdf = gpd.GeoDataFrame(flows, geometry=gpd.points_from_xy([p[0] for p in flows.start_point], [p[1] for p in flows.start_point]), crs=4326)
end_gdf = gpd.GeoDataFrame(flows, geometry=gpd.points_from_xy([p[0] for p in flows.end_point], [p[1] for p in flows.end_point]), crs=4326)



start_join = gpd.sjoin(start_gdf, zones_gdf, how="left")
start_join = start_join[~start_join.index.duplicated(keep='first')]
flows['origin_zone'] = start_join['name']


end_join = gpd.sjoin(end_gdf, zones_gdf, how="left")
end_join = end_join[~end_join.index.duplicated(keep='first')]
flows['dest_zone'] = end_join['name']

# spatial join된 flows를 aggregation을 통한 2차 요약
flow_summary = flows.groupby(['origin_zone', 'dest_zone'])['weight'].sum().reset_index()
flow_summary = flow_summary.sort_values(by='weight', ascending=False).head(13) # 상위 10개 항로

data_dict_flow = flow_summary.to_dict(orient='records')
trajs_flow_data = json.dumps(data_dict_flow, ensure_ascii=False, indent=4)

In [38]:
print(trajs_flow_data)

[
    {
        "origin_zone": "부산-가덕 해역",
        "dest_zone": "거제-가덕 외해",
        "weight": 16
    },
    {
        "origin_zone": "제주 남동부 해역",
        "dest_zone": "원거리 국제공해",
        "weight": 11
    },
    {
        "origin_zone": "거제-가덕 외해",
        "dest_zone": "부산-가덕 해역",
        "weight": 10
    },
    {
        "origin_zone": "시모노세키-기타큐슈 해역",
        "dest_zone": "시모노세키-기타큐슈 해역",
        "weight": 9
    },
    {
        "origin_zone": "거제-가덕 외해",
        "dest_zone": "여수-통영 외해",
        "weight": 9
    },
    {
        "origin_zone": "부산외항-울산남부",
        "dest_zone": "부산-가덕 해역",
        "weight": 8
    },
    {
        "origin_zone": "원거리 국제공해",
        "dest_zone": "제주 남동부 해역",
        "weight": 8
    },
    {
        "origin_zone": "여수-통영 외해",
        "dest_zone": "거제-가덕 외해",
        "weight": 8
    },
    {
        "origin_zone": "부산-가덕 해역",
        "dest_zone": "부산외항-울산남부",
        "weight": 7
    },
    {
        "origin_zone": "거제-가덕 외해",
        "dest_zone": "제주 남동부 해역

In [39]:
def get_flow_summary_duckdb(flows, zones_gdf):
    # 1. DuckDB 연결 및 공간 확장 로드
    con = duckdb.connect(database=':memory:')
    con.execute("INSTALL spatial; LOAD spatial;")

    # 2. 데이터 등록을 위한 WKB 변환
    # flows는 LineString geometry를 가짐
    flows['geom_wkb'] = flows.geometry.to_wkb()
    zones_gdf['geom_wkb'] = zones_gdf['geometry'].to_wkb()

    con.register('v_flows', flows[['weight', 'geom_wkb']])
    con.register('v_zones', zones_gdf[['name', 'geom_wkb']])

    # 3. 통합 SQL 쿼리
    # ST_StartPoint, ST_EndPoint 함수를 사용하여 좌표 추출 및 조인
    query = """
        WITH flow_base AS (
            SELECT 
                weight,
                ST_StartPoint(ST_GeomFromWKB(geom_wkb)) as start_pt,
                ST_EndPoint(ST_GeomFromWKB(geom_wkb)) as end_pt,
                row_number() OVER() as flow_id -- 각 항로에 고유 ID 부여
            FROM v_flows
        ),
        start_zones AS (
            -- 시작점당 구역 1개만 매칭 (기존 duplicated 제거 로직 재현)
            SELECT flow_id, name as origin_zone
            FROM (
                SELECT f.flow_id, z.name,
                       row_number() OVER(PARTITION BY f.flow_id ORDER BY z.name) as rn
                FROM flow_base f
                JOIN v_zones z ON ST_Intersects(f.start_pt, ST_GeomFromWKB(z.geom_wkb))
            ) WHERE rn = 1
        ),
        end_zones AS (
            -- 끝점당 구역 1개만 매칭
            SELECT flow_id, name as dest_zone
            FROM (
                SELECT f.flow_id, z.name,
                       row_number() OVER(PARTITION BY f.flow_id ORDER BY z.name) as rn
                FROM flow_base f
                JOIN v_zones z ON ST_Intersects(f.end_pt, ST_GeomFromWKB(z.geom_wkb))
            ) WHERE rn = 1
        )
        SELECT 
            s.origin_zone,
            e.dest_zone,
            SUM(f.weight) as weight
        FROM flow_base f
        JOIN start_zones s ON f.flow_id = s.flow_id
        JOIN end_zones e ON f.flow_id = e.flow_id
        GROUP BY s.origin_zone, e.dest_zone
        ORDER BY weight DESC
        LIMIT 13
    """

    # 4. 실행 및 결과 변환
    flow_summary_df = con.execute(query).df()
    
    if flow_summary_df.empty:
        return json.dumps([], ensure_ascii=False)

    return json.dumps(
        flow_summary_df.to_dict(orient='records'), 
        ensure_ascii=False, 
        indent=4
    )

In [40]:
start = time.time()
trajs_flow_data = get_flow_summary_duckdb(flows, zones_gdf)
print(f"소요시간: {time.time()-start}초")

소요시간: 0.06854724884033203초


In [41]:
print(trajs_flow_data)

[
    {
        "origin_zone": "부산-가덕 해역",
        "dest_zone": "거제-가덕 외해",
        "weight": 16.0
    },
    {
        "origin_zone": "제주 남동부 해역",
        "dest_zone": "원거리 국제공해",
        "weight": 11.0
    },
    {
        "origin_zone": "거제-가덕 외해",
        "dest_zone": "부산-가덕 해역",
        "weight": 10.0
    },
    {
        "origin_zone": "거제-가덕 외해",
        "dest_zone": "여수-통영 외해",
        "weight": 9.0
    },
    {
        "origin_zone": "시모노세키-기타큐슈 해역",
        "dest_zone": "시모노세키-기타큐슈 해역",
        "weight": 9.0
    },
    {
        "origin_zone": "원거리 국제공해",
        "dest_zone": "제주 남동부 해역",
        "weight": 8.0
    },
    {
        "origin_zone": "여수-통영 외해",
        "dest_zone": "거제-가덕 외해",
        "weight": 8.0
    },
    {
        "origin_zone": "부산외항-울산남부",
        "dest_zone": "부산-가덕 해역",
        "weight": 8.0
    },
    {
        "origin_zone": "거제-가덕 외해",
        "dest_zone": "제주 남동부 해역",
        "weight": 7.0
    },
    {
        "origin_zone": "원거리 국제공해",
        "dest

In [ ]:
# ShipType별 선박 Count 및 시각화
shiptype_count_summary = ais.groupby('ShipType')['mmsi'].nunique().sort_values(ascending=False)

#LLM을 위한 전처리
data_dict_shiptype = shiptype_count_summary.to_dict()
shiptype_data = json.dumps(data_dict_shiptype, ensure_ascii=False, indent=4)

In [49]:
zones_plot_data

,zone_id,name,geometry,mean,max,count
0,Zone_5,목포-신안 해역,"POLYGON ((126.6 34.4, 126.6 36, 125.5 36, 125....",8.364968,22.7,157
1,Zone_12,고흥-완도 연안,"POLYGON ((127.6 34.4, 127.6 36, 126.6 36, 126....",4.140000,10.0,5
2,Zone_3,광양-여수 해역,"POLYGON ((128 34.7, 128 36, 127.6 36, 127.6 34...",6.167031,15.7,458
3,Zone_2,거제-통영 해역,"POLYGON ((128.7 34.7, 128.7 36, 128.3 36, 128....",10.497297,12.8,37
4,Zone_1,부산-가덕 해역,"POLYGON ((129.3 34.7, 129.3 36, 128.7 36, 128....",2.370750,102.3,4653
5,Zone_18,부산외항-울산남부,"POLYGON ((130.5 34.7, 130.5 36, 129.3 36, 129....",7.930511,16.7,685
6,Zone_14,울산-포항 해역,"POLYGON ((132.5 35.15, 132.5 36, 130.5 36, 130...",11.489189,13.0,37
7,Zone_7,서남해 외해,"POLYGON ((126.2 33, 126.2 34.4, 125 34.4, 125 ...",16.158000,23.3,50
8,Zone_8,제주-육지 교차로 해역,"POLYGON ((127.6 33.8, 127.6 34.4, 126.2 34.4, ...",12.416923,21.8,130
9,Zone_9_1,여수-통영 외해,"POLYGON ((128.3 33.8, 128.3 34.7, 127.6 34.7, ...",5.068153,19.9,2176


In [50]:
start = time.time()
# 시각화용 데이터 병합 zone 데이터 + 속도 통계
zones_plot_data = zones_gdf.merge(zone_speed_stats, on='name')

#################################################
## 해상구역 + OD flow + 구역별 속도 평균 함께 가시화
#################################################
zones_plot_data['mean'] = pd.to_numeric(zones_plot_data['mean'])

# 구역  색상을 평균 속도로 구하기
zones_speed = zones_plot_data.hvplot(
    geo=True, tiles="OSM", alpha=0.45, c="mean",
    colorbar=True, clabel="Speed (knots)",
    line_color='black', line_width=1, cmap='YlOrRd', 
    hover_cols=['name', 'mean_knots'], 
    frame_height=400, frame_width=500, legend=False)

# 2. Aggregated Trajectories (MovingPandas 결과)
aggregated_chart = zones_speed * flows.hvplot(
    geo=True, hover_cols=['weight'], line_width='weight', 
    alpha=0.5, color='#1f77b3'
) * clusters.hvplot(
    geo=True, color='red', size='n'
)

# title 설정
flow_cluster_map = aggregated_chart.opts(
    title='Comparison: Aggregated OD Flows, Speed with Analysis Zones'
)
print(f"소요시간: {time.time()-start}초")
flow_cluster_map

소요시간: 0.02159905433654785초


:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]   (mean,name)
   .Path.I     :Path   [Longitude,Latitude]   (weight)
   .Points.I   :Points   [Longitude,Latitude]   (n)

In [ ]:
# 1. DuckDB 연결 및 공간 확장 로드 (기존 유지)
con = duckdb.connect(database=db_path)
con.execute("INSTALL spatial; LOAD spatial;")

# 2. 결과 데이터(zone_speed_stats)를 DuckDB에 등록
# 이미 계산된 통계치를 다시 DB로 보내 연산의 중심으로 삼습니다.
con.register('v_speed_stats', zone_speed_stats)

# [중요] zones_gdf는 이미 v_zones로 등록되어 있다고 가정합니다. 
# 만약 세션이 끊겼다면 아래 한 줄을 다시 실행하세요.
zones_gdf['geom_wkb'] = zones_gdf['geometry'].to_wkb()
con.register('v_zones', zones_gdf[['name', 'geom_wkb']])

# 3. DuckDB 통합 쿼리: 통계 데이터 + 구역 도형 데이터 병합
# 이 쿼리는 시각화에 필요한 모든 컬럼을 한 번에 정렬하여 반환합니다.
final_viz_query = """
    SELECT 
        z.name,
        z.geom_wkb,
        CAST(s.mean AS DOUBLE) as mean,
        s.max,
        s.count
    FROM v_zones z
    INNER JOIN v_speed_stats s ON z.name = s.name
    ORDER BY s.mean DESC
"""

# 4. 결과 실행 및 GeoDataFrame 복원
viz_df = con.execute(final_viz_query).df()

# DuckDB에서 가져온 WKB를 다시 Shapely geometry로 변환 (GeoPandas 최적화)
zones_plot_data = gpd.GeoDataFrame(
    viz_df,
    geometry=gpd.GeoSeries.from_wkb(viz_df['geom_wkb']),
    crs="EPSG:4326"
).drop(columns=['geom_wkb'])

#################################################
## 최적화된 시각화 실행 (전달받은 viz_df 사용)
#################################################

# 1. 해상구역 배경 (평균 속도 기반 채색)
zones_speed = zones_plot_data.hvplot(
    geo=True, tiles="OSM", alpha=0.45, c="mean",
    colorbar=True, clabel="Avg Speed (knots)",
    line_color='black', line_width=1, cmap='YlOrRd', 
    hover_cols=['name', 'mean', 'count'], # mean_knots 대신 mean 사용
    frame_height=400, frame_width=500, legend=False,
    responsive=True # 이전의 레이아웃 어긋남 방지
)

# 2. 레이어 통합
flow_cluster_map = (zones_speed * flows.hvplot(geo=True, hover_cols=['weight'], line_width='weight', alpha=0.5, color='#1f77b3') * clusters.hvplot(geo=True, color='red', size='n')
).opts(
    title='Integrated Analysis: OD Flows, Avg Speed & Clusters',
    active_tools=['wheel_zoom']
)

# 3. HTML 변환 및 전송 준비 (이전 로직 활용)
renderer = hv.renderer('bokeh')
plot_state = renderer.get_plot(flow_cluster_map).state
plot_state.sizing_mode = 'stretch_both' # 레이아웃 최적화